## TEST AND FIX FuckingFAST

In [ ]:
"""
fuckingfast.co link extractor — LOCAL (Windows/desktop) version.

Why local + why a persistent Chrome profile:
fuckingfast.co guards the download with a Cloudflare Turnstile. The DOWNLOAD
button only fires its POST when window.turnstileToken is set:

    hx-trigger="click[!!window.turnstileToken || !!window.dlCleared]"
    hx-vals="js:{'cf-turnstile-response': window.turnstileToken}"

Turnstile refuses to issue that token inside an *ephemeral* automation browser
(fresh Playwright context) — it flags it as a bot, so the click never posts.
The fix is to drive a REAL Chrome with a PERSISTENT user-data dir
(launch_persistent_context + channel="chrome"). Then Turnstile mints the token
silently in ~3s, and the second click returns the CDN link via hx-redirect.

No visible window: Chrome is started MINIMIZED (--start-minimized) with window
occlusion/throttling disabled, so it still renders (Turnstile passes) but never
pops up on screen. Fully headless does NOT work — Turnstile detects it.

Skipping the manual checkbox: we only wait TOKEN_TIMEOUT seconds for the token.
If Cloudflare decides to show the interactive "verify you are human" checkbox
(token never auto-arrives), we skip that link instead of blocking on it.

One-time setup (run in a terminal):
    pip install playwright curl_cffi cryptography
    playwright install chromium      # only needed if you don't have Chrome

Put your links in LINK_INPUTS below. Supports:
  * https://fuckingfast.co/<id>#name.rar          (direct file pages)
  * https://paste.fitgirl-repacks.site/?...#key    (decrypted, then its ff links)
Results are printed and written to fuckingfast_links.txt.
"""

import os
import re
import sys
import json
import zlib
import base64
import asyncio

from curl_cffi import requests as cffi_requests
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from playwright.async_api import async_playwright

# Persistent Chrome profile lives next to this file — reused across runs so
# Turnstile keeps treating us as a returning, real browser.
_HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
PROFILE_DIR = os.path.join(_HERE, ".ff_profile")

# How long to wait for Turnstile to auto-issue the token before giving up on a
# link (so we don't hang when the manual checkbox appears).
TOKEN_TIMEOUT = 30

# Chrome flags: real headful render (needed for Turnstile) but window is
# minimized and never throttled, so nothing shows on screen.
_CHROME_ARGS = [
    "--disable-blink-features=AutomationControlled",
    "--start-minimized",
    "--disable-features=CalculateNativeWinOcclusion",
    "--disable-backgrounding-occluded-windows",
    "--disable-renderer-backgrounding",
    "--disable-background-timer-throttling",
]

# ── PrivateBin helpers ────────────────────────────────────────────────────────
_BASE58 = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'


def _base58_decode(s):
    num = 0
    for ch in s:
        num = num * 58 + _BASE58.index(ch)
    result = b''
    while num > 0:
        num, rem = divmod(num, 256)
        result = bytes([rem]) + result
    return b'\x00' * (len(s) - len(s.lstrip('1'))) + result


def decrypt_privatebin(url, password=''):
    """Decrypt a PrivateBin v2 paste. The key lives in the URL fragment."""
    paste_url, url_key = url.split('#', 1)
    paste_id = paste_url.split('?')[-1]
    base_url = paste_url[:paste_url.index('?')]

    res = cffi_requests.get(f"{base_url}?{paste_id}",
                            headers={"Accept": "application/json"},
                            impersonate="chrome", timeout=60)
    data = res.json()
    if data.get('status') != 0:
        raise ValueError(f"PrivateBin API error: {data}")

    adata = data['adata']
    iv_b64, salt_b64, iterations, key_size, _tag, _algo, _mode, compression = adata[0]
    iv = base64.b64decode(iv_b64)
    salt = base64.b64decode(salt_b64)
    ct = base64.b64decode(data['ct'])

    key_input = _base58_decode(url_key)
    if password:
        key_input += password.encode()

    derived = PBKDF2HMAC(hashes.SHA256(), key_size // 8, salt, iterations).derive(key_input)
    aad = json.dumps(adata, separators=(',', ':')).encode()
    raw = AESGCM(derived).decrypt(iv, ct, aad)
    if compression == 'zlib':
        raw = zlib.decompress(raw, wbits=-15)
    payload = json.loads(raw.decode())
    return payload.get('paste', raw.decode())


# ── fuckingfast.co extraction via a REAL persistent Chrome ────────────────────
async def _extract_all(urls, token_timeout=TOKEN_TIMEOUT, per_url_timeout=90):
    extracted = []
    async with async_playwright() as p:
        # Persistent, real-Chrome context — this is what makes Turnstile pass.
        # headless is intentionally False; the window is minimized instead.
        ctx = await p.chromium.launch_persistent_context(
            PROFILE_DIR,
            headless=False,
            channel="chrome",
            args=_CHROME_ARGS,
            accept_downloads=False,   # following the CDN link must never pull the file
            no_viewport=True,
        )
        page = ctx.pages[0] if ctx.pages else await ctx.new_page()

        # fuckingfast opens an ad tab on the first click — close any extra tab.
        async def _close_popup(pg):
            if pg is page:
                return
            try:
                await pg.close()
            except Exception:
                pass
        ctx.on("page", lambda pg: asyncio.create_task(_close_popup(pg)))

        for url in urls:
            fid = url.split('#')[0].rstrip('/').split('/')[-1]
            got = {}

            async def on_resp(r, fid=fid, got=got):
                if r.request.method == "POST" and r.url.rstrip('/').endswith(f"/f/{fid}/go"):
                    loc = await r.header_value("hx-redirect")
                    if loc:
                        got["link"] = loc

            page.on("response", on_resp)
            try:
                await page.goto(f"https://fuckingfast.co/{fid}",
                                wait_until="domcontentloaded",
                                timeout=per_url_timeout * 1000)
                # Wait (briefly) for Turnstile to auto-issue window.turnstileToken.
                got_token = True
                try:
                    await page.wait_for_function(
                        "() => !!window.turnstileToken || !!window.dlCleared",
                        timeout=token_timeout * 1000)
                except Exception:
                    got_token = False

                if not got_token:
                    # Checkbox likely showed up — don't hang on it, skip this link.
                    print(f"  \u21b7 skipped {fid} (Turnstile needs the manual checkbox)")
                    continue

                # First click opens the ad tab, the next one fires the download POST.
                btn = page.locator("a[hx-post]")
                for _ in range(5):
                    if "link" in got:
                        break
                    try:
                        await btn.click(timeout=5000, no_wait_after=True)
                    except Exception:
                        pass
                    for _ in range(20):
                        if "link" in got:
                            break
                        await asyncio.sleep(0.5)
            except Exception as e:
                print(f"  Error for {fid}: {type(e).__name__}: {e}")
            finally:
                page.remove_listener("response", on_resp)

            if "link" in got:
                print(f"  \u2713 {got['link']}")
                extracted.append(got["link"])
            else:
                print(f"  \u2717 no link for {fid} (Turnstile not cleared)")

        await ctx.close()
    return extracted


def _run_async(coro):
    """Run an async coroutine, whether or not an event loop is already running.

    In a plain script we can use asyncio.run(). Inside Jupyter/IPython a loop is
    already running, so we run the coroutine in a dedicated thread with its own
    fresh loop. On Windows that loop must be a ProactorEventLoop, otherwise
    Playwright can't spawn its browser subprocess (Jupyter/Tornado installs a
    SelectorEventLoop policy that raises NotImplementedError on subprocesses).
    """
    def _thread_run():
        if sys.platform == "win32":
            loop = asyncio.ProactorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.close()

    try:
        asyncio.get_running_loop()
    except RuntimeError:
        # No running loop → we can run in this thread directly.
        return _thread_run()

    # A loop is already running (Jupyter/IPython) → offload to a worker thread.
    import threading
    box = {}

    def _runner():
        try:
            box["value"] = _thread_run()
        except BaseException as e:  # noqa: BLE001 - surface any failure to caller
            box["error"] = e

    t = threading.Thread(target=_runner)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


def extract_download_links_from_pages(urls, token_timeout=TOKEN_TIMEOUT):
    return _run_async(_extract_all(urls, token_timeout=token_timeout))


# ── your links ────────────────────────────────────────────────────────────────
LINK_INPUTS = '''
# One link per line. Lines starting with # are ignored.
# Paste fuckingfast.co links or a paste.fitgirl-repacks.site link.

https://paste.fitgirl-repacks.site/?b7f1f779cba7bfbc#GupC3qoJmtxSCbKUriCn5WoMMhVbExvAzLS4Y4Y5M79U
'''


def main():
    links = [x.strip() for x in LINK_INPUTS.splitlines()
             if x.strip() and not x.strip().startswith('#')]

    all_results = []
    for link_input in links:
        if link_input.startswith('https://paste.fitgirl'):
            try:
                markdown = decrypt_privatebin(link_input)
                urls = re.findall(r'https://fuckingfast\.co/\S+', markdown)
            except Exception as e:
                print(f"  Error decrypting {link_input}: {e}. Skipping.")
                continue
        elif link_input.startswith('https://fuckingfast.co'):
            urls = [link_input]
        else:
            print(f"  Unsupported link: {link_input}")
            continue

        if urls:
            print(f"\nProcessing {len(urls)} url(s)...")
            all_results.extend(extract_download_links_from_pages(urls))

    print("\n================ RESULTS ================")
    for r in all_results:
        print(r)

    if all_results:
        with open("fuckingfast_links.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(all_results))
        print(f"\nSaved {len(all_results)} link(s) to fuckingfast_links.txt")


if __name__ == "__main__":
    main()



Processing 27 url(s)...
  ↷ skipped yjoj5ds9m14e (Turnstile needs the manual checkbox)
  ↷ skipped nis0jv8ckvje (Turnstile needs the manual checkbox)
  ↷ skipped n38fzkl0805c (Turnstile needs the manual checkbox)
  ↷ skipped 7y8hyav2l6o2 (Turnstile needs the manual checkbox)
  ↷ skipped 5xw8s181jkce (Turnstile needs the manual checkbox)
  ↷ skipped p0z5pqbhajgu (Turnstile needs the manual checkbox)
  ↷ skipped m9z6byxtutby (Turnstile needs the manual checkbox)
  ↷ skipped 4gdjfn8ew0qs (Turnstile needs the manual checkbox)
  ↷ skipped mp4xena3bx9y (Turnstile needs the manual checkbox)
  ↷ skipped 3km1f16jp5cr (Turnstile needs the manual checkbox)


## TEST filekeeper

### link will be lik either

```https://filekeeper.net/5y9fgeiiawb2/ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part01.rar```

or 

```https://paste.fitgirl-repacks.site/?9b83a394a1d920a1#HNJUSConLNWoHU8vdkuAaHPzjQuAe8C3tShNzbWNTCT9
```

but here is the way it work 

you open this 
https://filekeeper.net/5y9fgeiiawb2/ACBF_Resynced_CRACKED_--_fitgirl-repacks.site_--_.part01.rar

it goes to this 

https://filekeeper.net/download

you wait 5 sec it will appear like 

[text](HTMLS/filekeeper.html)

you get link like that 

https://tunnel1.dlproxy.uk/download/j974CvvJb3EFHa_LkHxUOdqg9cltPioZ1XaNcnrwKcdHPye7wX0w7hD_Ko0Y2z77-lSIW5n0w2SygpXbsYvnGX0gfznaZP5sgLoVK6MGje1ZD5qyvmRKT1mp0DBLzfSv3WRTNlk8yF2p6cQX31pLcCcQMu9BbPeSUU2t3R1QHtmzIv3adr0SajqAk7G0j6m09apAxoeJ8ecav1Q1br-jRoRRYMMlqYPQlTpEciThbTbQ72U8LvV_6mGze7MU9pZi5wdsDiWb0kW5kMki9TwJxm3-37cITcOYskOT56Qh-U1C0L7EnKf1s3WG3VW8QctLHtvpY5_3vmYbTwB0nvXuTwgcUa1VBtBXdB8ZPGrWjwiysBYPkWj7AFmzHXqlYcY9QxKHDXxnQ0fiaa4va6QLuDMDd9EgiLZOoGdRZ2vOuToBJvs1erP02Q74O1-kzJDq69FBvrLMTjVFb_wgCc6x-S3kL_TnsNKBoiYxOnJs6ClHTDsOae1Zr2Of_5kr1Gy1aQxFeWEqOFurSd0-KJLorYPscKDAoP9qpQvHmyeZBe1qbdcKXlV0UA_MbAuP5Q_NfypmDMNbNcj8GgKplAg1z8lkZr8vP2t8FPVLNBB_Y7oxon8ZKkEylTbjR9TJZN0C-VJsfpXy0e4mIbMosKX3mw?sig=KwzhiOIca0sHqJUQA2SmP4sYukcKbsW0DT0h78N6L5c	


